In [ ]:
%cd /content/
!rm -rf OmniFile_Processor
!git clone https://github.com/DrAbdulmalek/OmniFile_Processor.git
%cd OmniFile_Processor
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq poppler-utils tesseract-ocr tesseract-ocr-ara tesseract-ocr-eng libgl1 > /dev/null 2>&1
!pip install -r requirements-colab.txt > /dev/null 2>&1

# Fix for Gradio TypeError: File.__init__() got an unexpected keyword argument 'multiple'
!sed -i "s/multiple=True/file_count='multiple'/g" hf_app.py

# Fix for Dataframe height warning in src/correction_trainer_ui.py (optional, but good to include)
!sed -i '/height\s*=\s*150/d' src/correction_trainer_ui.py

# Create modify_hf_app.py using Python's file write within the cell
# This avoids shell here-document parsing issues.
modify_hf_app_content = """
import re
import os

file_path = 'hf_app.py'

if not os.path.exists(file_path):
    print("Error: {} not found. Cannot apply modifications.".format(file_path))
else:
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Fix for Gradio Dataframe.__init__() got an unexpected keyword argument 'height'
    # Robustly remove 'height = NNN' and surrounding commas.
    content = re.sub(r',\\s*height\\s*=\\s*\\d+', '', content)
    content = re.sub(r'height\\s*=\\s*\\d+\\s*,', '', content)
    content = re.sub(r'height\\s*=\\s*\\d+', '', content) # Catch isolated 'height=NNN'

    # Function to process the arguments of demo.launch()
    def process_launch_args(match):
        args_content = match.group(1) # Content inside demo.launch(...)

        # Remove theme argument. This regex needs to handle various theme calls.
        # It removes `theme=gr.themes.XYZ(...)` and any preceding comma.
        # New, more robust regex to remove the entire theme block
        args_content = re.sub(
            r',\\s*theme\\s*=\\s*gr\\.themes\\.\\w+\\((?:[^)]|\\s)*\\),?', '', args_content, flags=re.DOTALL
        )
        args_content = re.sub(
            r'theme\\s*=\\s*gr\\.themes\\.\\w+\\((?:[^)]|\\s)*\\),?\\s*,', '', args_content, flags=re.DOTALL
        )

        # Remove any existing share argument
        args_content = re.sub(r',\\s*share\\s*=\\s*(True|False)', '', args_content)
        args_content = re.sub(r'share\\s*=\\s*(True|False)\\s*,', '', args_content)

        # Cleanup double commas and leading/trailing commas after removals
        args_content = re.sub(r',\\s*,', ',', args_content)
        args_content = args_content.strip()
        args_content = re.sub(r'^\\s*,\\s*', '', args_content)
        args_content = re.sub(r',\\s*$', '', args_content)

        # Add share=True if not present
        if 'share=True' not in args_content:
            if args_content:
                args_content += ", share=True"
            else:
                args_content = "share=True"

        return f"demo.launch({args_content})"

    # Find the demo.launch call and apply the processing function
    content = re.sub(r'demo\\.launch\\((.*?)\\)', process_launch_args, content, flags=re.DOTALL)

    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("hf_app.py modified successfully by Python script.")
"""

# Write the content to the file
with open('modify_hf_app.py', 'w') as f:
    f.write(modify_hf_app_content)

# Run the Python modification script
!python modify_hf_app.py

# Debugging: Print the file content before execution
!cat hf_app.py

# Run the Gradio application
!python hf_app.py

/content
Cloning into 'OmniFile_Processor'...
remote: Enumerating objects: 1119, done.
remote: Counting objects: 100% (246/246), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 1119 (delta 86), reused 204 (delta 66), pack-reused 873 (from 1)
Receiving objects: 100% (1119/1119), 35.16 MiB | 15.84 MiB/s, done.
Resolving deltas: 100% (417/417), done.
/content/OmniFile_Processor


In [26]:
%%writefile modify_hf_app.py
import re

file_path = 'hf_app.py'

with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Fix for Gradio Dataframe.__init__() got an unexpected keyword argument 'height'
# Robustly remove 'height = NNN' and surrounding commas.
content = re.sub(r',\s*height\s*=\s*\d+', '', content)
content = re.sub(r'height\s*=\s*\d+\s*,', '', content)
content = re.sub(r'height\s*=\s*\d+', '', content) # Catch isolated 'height=NNN'

# Function to process the arguments of demo.launch()
def process_launch_args(match):
    args_content = match.group(1) # Content inside demo.launch(...)

    # Remove theme argument. This regex needs to handle various theme calls.
    # It removes `theme=gr.themes.XYZ(...)` and any preceding comma.
    args_content = re.sub(r',\s*theme\s*=\s*gr\.themes\.\w+\([^)]*\)', '', args_content, flags=re.DOTALL)
    args_content = re.sub(r'theme\s*=\s*gr\.themes\.\w+\([^)]*\)\s*,', '', args_content, flags=re.DOTALL)

    # Remove any existing share argument
    args_content = re.sub(r',\s*share\s*=\s*(True|False)', '', args_content)
    args_content = re.sub(r'share\s*=\s*(True|False)\s*,', '', args_content)

    # Cleanup double commas and leading/trailing commas after removals
    args_content = re.sub(r',\s*,', ',', args_content)
    args_content = args_content.strip()
    args_content = re.sub(r'^\s*,\s*', '', args_content)
    args_content = re.sub(r',\s*$', '', args_content)

    # Add share=True if not present
    if 'share=True' not in args_content:
        if args_content:
            args_content += ", share=True"
        else:
            args_content = "share=True"

    return f"demo.launch({args_content})"

# Find the demo.launch call and apply the processing function
content = re.sub(r'demo\.launch\((.*?)\)', process_launch_args, content, flags=re.DOTALL)

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(content)

print("hf_app.py modified successfully by Python script.")

Overwriting modify_hf_app.py
